<a href="https://colab.research.google.com/github/zwimpee/cursivetransformer/blob/main/visualize_attention.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Clone the cursivetransformer repository and install its requirements
!rm -rf cursivetransformer && git clone https://github.com/zwimpee/cursivetransformer.git
!cd cursivetransformer && pip install -r requirements.txt
!wandb login

Cloning into 'cursivetransformer'...
remote: Enumerating objects: 3051, done.
remote: Counting objects: 100% (665/665), done.
remote: Compressing objects: 100% (180/180), done.
remote: Total 3051 (delta 594), reused 517 (delta 485), pack-reused 2386 (from 1)
Receiving objects: 100% (3051/3051), 64.70 MiB | 38.76 MiB/s, done.
Resolving deltas: 100% (1708/1708), done.
  Cloning https://github.com/callummcdougall/CircuitsVis.git to /tmp/pip-req-build-rn322i_c
  Running command git clone --filter=blob:none --quiet https://github.com/callummcdougall/CircuitsVis.git /tmp/pip-req-build-rn322i_c
  Resolved https://github.com/callummcdougall/CircuitsVis.git to commit 1e6129d08cae7af9242d9ab5d3ed322dd44b4dd3
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.6/50.6 kB 4.5 MB/s eta 0:00:00
INFO: pip is looking at multiple versions of multiprocess to determine which 

In [2]:
import sys
sys.path.append('/content/cursivetransformer/')

import torch
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")

from cursivetransformer.model import get_all_args, get_checkpoint
from cursivetransformer.data import create_datasets
from cursivetransformer.sample import generate_n_words, plot_strokes

In [3]:
args = get_all_args(False)
args.wandb_project = 'bigbank_2k'
args.load_from_run_id = '7e9hz1og'
args.max_seq_length = 1250
torch.manual_seed(args.seed)
torch.cuda.manual_seed_all(args.seed)
train_dataset, test_dataset = create_datasets(args)
args.block_size = train_dataset.get_stroke_seq_length()
args.context_block_size = train_dataset.get_text_seq_length()
args.vocab_size = train_dataset.get_vocab_size()
args.context_vocab_size = train_dataset.get_char_vocab_size()
print(f"Dataset determined that: {args.vocab_size=}, {args.block_size=}")
model, _, _, _, _ = get_checkpoint(args, sample_only=True)

Enter your W&B API key: ··········
Trying to load dataset file from /content/cursivetransformer/data/bigbank.json.zip
Succeeded in loading the bigbank dataset; contains 2000 items.
For a dataset of 1900 examples we can generate 205257574037880 combinations of 5 examples.
Generating 497000 random combinations.
For a dataset of 100 examples we can generate 75287520 combinations of 5 examples.
Generating 3000 random combinations.
Number of examples in the train dataset: 497000
Number of examples in the test dataset: 3000
Max token sequence length: 1250
Number of unique characters in the ascii vocabulary: 71
Ascii vocabulary:
	" enaitoshrdx.vpukbgfcymzw1lqj804I92637OTAS5N)EHR"'(BCQLMWYU,ZF!DXV?KPGJ"
Split up the dataset into 497000 training examples and 3000 test examples
Dataset determined that: args.vocab_size=382, args.block_size=1250
Number of Transformer parameters: 379392
Model #params: 403840
Finding latest checkpoint for W&B run id 7e9hz1og
  model:best_checkpoint:v129
  model:best

wandb: Using wandb-core as the SDK backend. Please refer to https://wandb.me/wandb-core for more information.
wandb:   1 of 1 files downloaded.  


In [10]:
x, c, y = test_dataset[0]
x = x.unsqueeze(0).to('cuda')
c = c.unsqueeze(0).to('cuda')
y = y.unsqueeze(0).to('cuda')

In [11]:
# Assuming 'model' is your Transformer instance

# Attach a hook to the first self-attention layer
self_attn_layer = model.transformer.h[0].attn
self_attn_patterns = {}

self_attn_layer.register_forward_hook(
    lambda mod, inp, out: self_attn_patterns.update({mod: out.detach()})
)

# Attach a hook to the first cross-attention layer
cross_attn_layer = model.transformer.h[0].cross_attn
cross_attn_patterns = {}

cross_attn_layer.register_forward_hook(
    lambda mod, inp, out: cross_attn_patterns.update({mod: out.detach()})
)

# Now run your model
with torch.no_grad():
  output = model(x, c)

# Access the attention patterns
print("Self-attention pattern shape:", next(iter(self_attn_patterns.values())).shape)
print("Cross-attention pattern shape:", next(iter(cross_attn_patterns.values())).shape)

IndexError: index 1 is out of bounds for dimension 0 with size 1